# 항 중요도 분석 - 경로별 기여도 측정

- Tutorial ID: `adv-5-1`
- Tutorial: 항 중요도 분석
- Section ID: `adv-5-1-1`
- Section: 경로별 기여도 측정

## 이 노트북에서 배우는 것

트랜스포머는 흔히 "블랙박스"라고 불리지만, 사실 모델이 왜 특정 토큰을 예측했는지를 수학적으로 정확하게 쪼개서 설명할 수 있는 방법들이 있습니다. 이런 분석 방법을 흔히 기계적 해석가능성(mechanistic interpretability)이라고 부릅니다.

이 노트북에서는 작은 장난감(toy) 트랜스포머를 직접 만들고, 그 모델이 내놓은 하나의 예측을 다음 세 가지 도구로 "분해"해 봅니다.

1. **직접 로짓 기여 (Direct Logit Attribution, DLA)** — 모델의 각 구성 요소(직접 경로, 각 attention head)가 최종 예측 점수에 정확히 얼마만큼을 더했는지 계산합니다.
2. **합성 점수 (composition score)** — 서로 다른 층(layer)에 있는 두 헤드가 "팀플레이"를 하고 있는지, 즉 한 헤드의 출력을 다른 헤드가 읽어서 활용하고 있는지를 측정합니다.
3. **어블레이션 (ablation, 제거 실험)** — 특정 구성 요소를 실제로 빼 보고 예측이 얼마나 흔들리는지 관찰해서, 그 구성 요소가 정말로 필요했는지를 인과적으로 확인합니다.

**주의:** 이 노트북에서 사용하는 가중치는 전부 `np.random.seed(42)`로 만든 무작위 값이며, 학습(training)은 전혀 진행하지 않습니다. 즉 모델이 똑똑해서 좋은 답을 내놓을 거라 기대하면 안 됩니다. 이 노트북의 목적은 "모델이 무엇을 예측하든, 그 예측이 내부적으로 어떻게 만들어졌는지 정확히 추적하는 도구"를 직접 만들어 보는 것입니다.

### 미리 알고 있으면 좋은 것
- 행렬곱(`@`)과 numpy 배열 연산에 대한 기본적인 이해
- attention의 대략적인 동작 (Q/K/V를 만들고, 점수를 매기고, softmax로 가중합을 만드는 흐름). 이 노트북에서도 다시 짚고 넘어가지만, 처음 본다면 이전 튜토리얼을 먼저 보는 것을 권장합니다.

## 핵심 아이디어: 왜 "항(項)"별로 쪼갤 수 있을까?

이 노트북 전체는 사실 하나의 간단한 수학적 사실 위에 세워져 있습니다.

> 세 값 a, b, c를 먼저 더한 뒤 선형 함수 f를 적용한 결과는, a, b, c를 각각 f에 통과시킨 다음 더한 결과와 정확히 같습니다.
>
> `f(a + b + c) = f(a) + f(b) + f(c)`

트랜스포머의 **잔차 스트림(residual stream)** 은 정확히 이 구조를 따릅니다. 임베딩, 그리고 각 층의 각 attention head는 정보를 "대체"하지 않고 잔차 연결(residual connection)을 통해 기존 값 위에 "더하기"만 합니다. 그리고 마지막에 다음 토큰의 점수(logit)를 계산하는 언임베딩(unembedding, `W_U`를 곱하는 연산)은 행렬곱이므로 선형 함수입니다.

그 결과, 최종 logit은 다음과 같이 정확히 분해됩니다.

```
logits = (직접경로 + L0H0출력 + L0H1출력 + L1H0출력 + L1H1출력) @ W_U
       = (직접경로 @ W_U) + (L0H0출력 @ W_U) + (L0H1출력 @ W_U) + (L1H0출력 @ W_U) + (L1H1출력 @ W_U)
```

각 항을 따로 계산해서 더하면 정확히 원래 logit과 같습니다 (근사가 아니라 **정확히** 같습니다). 이 노트북의 "항 중요도 분석"이라는 제목은 바로 이 각각의 항(직접경로, L0H0, L0H1, L1H0, L1H1)이 최종 예측에 얼마나 기여했는지를 살펴본다는 뜻입니다.

먼저 아주 작은 숫자 예제로 이 성질이 실제로 성립하는지 직접 확인해 봅시다.

(참고: 실제 트랜스포머에는 MLP 층과 LayerNorm도 있습니다. MLP도 잔차 스트림에 "더하는" 구조라 같은 논리가 적용되지만, LayerNorm은 비선형적인 정규화 과정이 섞여 있어서 엄밀하게는 완전한 선형 분해가 깨집니다. 실제 해석가능성 연구에서는 LayerNorm을 근사적으로 선형 변환처럼 취급하고 분석하는 경우가 많습니다. 이 노트북의 장난감 모델에는 MLP와 LayerNorm을 아예 넣지 않았기 때문에, 분해가 100% 정확하게 들어맞는 것을 직접 눈으로 확인할 수 있습니다.)

In [ ]:
import numpy as np

# -----------------------------------------------------------------
# 미니 데모: "더한 뒤 통과" == "각각 통과한 뒤 더하기" 확인하기
# -----------------------------------------------------------------
# 본격적인 트랜스포머 예제에 들어가기 전에, 핵심 아이디어가 실제로
# 성립하는지 아주 작은 숫자로 먼저 확인해 봅니다.

a = np.array([1.0, 2.0])   # 어떤 벡터 a
b = np.array([3.0, -1.0])  # 어떤 벡터 b
c = np.array([0.5, 0.5])   # 어떤 벡터 c

# 임의의 "선형 함수" 역할을 할 2x2 행렬 (그냥 곱셈일 뿐입니다)
W_demo = np.array([
    [2.0, 0.0],
    [0.0, 3.0],
])

# 방법 1: 먼저 a, b, c를 다 더한 다음에 W_demo를 곱한다
left = (a + b + c) @ W_demo

# 방법 2: a, b, c를 각각 따로 W_demo에 곱한 다음 결과를 더한다
right = (a @ W_demo) + (b @ W_demo) + (c @ W_demo)

print("방법 1, (a+b+c) @ W   :", left)
print("방법 2, a@W+b@W+c@W   :", right)
print("두 결과가 완전히 같은가?  ", np.allclose(left, right))

## 이 노트북의 구성

아래 순서대로 셀을 실행하면서 따라가면 됩니다.

| 구간 | 내용 |
|---|---|
| 0. 준비 | 하이퍼파라미터, 토이(toy) 어휘, 임베딩, 입력 문장, attention 가중치를 만듭니다 |
| 1. 전방 패스 | 2층(layer) x 2헤드(head) attention만으로 이루어진 아주 작은 모델을 직접 순전파(forward pass)시켜 다음 토큰을 예측합니다 |
| 2. 직접 로짓 기여 (DLA) | 예측에 각 구성 요소가 얼마나 기여했는지 "분해"해서 봅니다 |
| 3. 합성 점수 | layer 0의 헤드 출력을 layer 1의 헤드가 실제로 "읽고" 있는지 측정합니다 |
| 4. 어블레이션 | 구성 요소를 실제로 제거해 보면서 인과적인 중요도를 확인합니다 |

코드를 읽을 때는 다음을 함께 보는 것을 권장합니다.

- 각 줄 옆/위의 주석: 그 줄이 무엇을, 왜 하는지 설명합니다.
- `print`로 출력되는 shape과 값: 숫자가 실제로 어떻게 변해가는지 눈으로 확인하는 용도입니다.
- 직접 숫자를 바꿔보기: `seed`, `seq`(입력 문장), `d_head`, `n_heads` 등을 바꿔보면서 결과가 어떻게 달라지는지 실험해 보면 이해에 큰 도움이 됩니다.

In [ ]:
import numpy as np

print("=" * 60)
print("항 중요도 분석 - 경로별 기여도 측정")
print("=" * 60)


def softmax(x, axis=-1):
    '''
    점수(logit) 배열을 '합이 1인 확률 분포'로 바꿔주는 함수입니다.

    np.exp(x)를 곧바로 쓰지 않고 x - max(x)를 먼저 빼는 이유:
    x 값이 크면 np.exp(x)가 너무 커져서(overflow) 계산이 깨질 수 있습니다.
    e^(x-max) 형태로 만들면 가장 큰 값이 e^0 = 1이 되어 수치적으로 안전해지고,
    softmax 결과 자체는 수학적으로 원래 식과 완전히 동일합니다.
    '''
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / np.sum(e_x, axis=axis, keepdims=True)


# 간단한 동작 확인: 점수가 클수록 더 큰 확률을 받는지 확인
demo_scores = np.array([1.0, 3.0, 2.0])
print("\n[softmax 동작 확인]")
print("입력 점수:", demo_scores)
print("출력 확률:", np.round(softmax(demo_scores), 4), " (합:", softmax(demo_scores).sum(), ")")

## 하이퍼파라미터가 의미하는 것

작은 모델을 만들기 전에, 앞으로 계속 등장할 숫자들이 각각 무엇을 의미하는지 먼저 정리합니다.

- `vocab_size` : 모델이 알고 있는 전체 단어(토큰)의 개수입니다. 이 노트북에서는 8개의 단어만 사용하는 아주 작은 장난감 어휘를 만듭니다.
- `d_model` : 잔차 스트림의 차원입니다. 각 토큰은 이 차원의 벡터로 표현되며, 모든 층의 입출력이 이 차원을 공유합니다.
- `d_head` : attention head 하나가 내부적으로 사용하는 (더 작은) 차원입니다. `d_model`보다 작은 병목(bottleneck)을 거치게 해서, 각 head가 잔차 스트림 전체가 아니라 일부 정보만 골라서 읽고 쓰도록 만드는 역할을 합니다.
- `n_heads` : 한 층(layer)에 들어 있는 attention head의 개수입니다.
- `n_layers` : attention 층의 개수입니다. (이 노트북에서는 MLP 없이 attention layer만 2개 쌓습니다.)

이 노트북에서는 `vocab_size=8, d_model=12, d_head=6, n_heads=2, n_layers=2` 로 설정합니다. 실제 LLM은 이 숫자들이 수천~수만 단위지만, 동작 원리는 완전히 동일합니다.

In [ ]:
# -----------------------------------------------------------------
# 하이퍼파라미터
# -----------------------------------------------------------------
vocab_size = 8    # 어휘 크기 (토큰 종류 수)
d_model = 12      # 잔차 스트림 차원
d_head = 6        # 헤드 하나의 내부 차원 (d_model보다 작은 병목)
n_heads = 2       # 층당 head 개수
n_layers = 2      # attention 층 개수

np.random.seed(42)  # 재현 가능하도록 시드 고정 (같은 코드를 실행하면 항상 같은 '무작위' 값이 나옵니다)

# -----------------------------------------------------------------
# 토이 어휘: 8개의 한국어 단어에 토큰 id(0~7)를 붙입니다
# -----------------------------------------------------------------
# 실제 모델의 토큰화(tokenization)는 훨씬 복잡하지만, 여기서는
# "토큰 id 하나 = 단어 하나"인 아주 단순한 장난감 어휘를 사용합니다.
id_to_word = {
    0: "나는",
    1: "오늘",
    2: "학교에",
    3: "간다",
    4: "온다",
    5: "집에",
    6: "어제",
    7: "잤다",
}
print("[토이 어휘]")
for i, w in id_to_word.items():
    print(f"   token {i}: {w}")

# -----------------------------------------------------------------
# 임베딩 / 언임베딩 행렬
# -----------------------------------------------------------------
# W_E (embedding): 토큰 id -> d_model 차원 벡터로 '올려보내는' 행렬
#   - shape: (vocab_size, d_model) = (8, 12)
#   - W_E[i]가 바로 토큰 i의 임베딩 벡터입니다.
W_E = np.random.randn(vocab_size, d_model) * 0.3

# W_U (unembedding): d_model 차원 벡터 -> vocab_size개의 점수(logit)로
# '내려보내는' 행렬. 모델이 다음 토큰 후보 8개 각각에 점수를 매기는 단계입니다.
#   - shape: (d_model, vocab_size) = (12, 8)
W_U = np.random.randn(d_model, vocab_size) * 0.2

print("\nW_E shape:", W_E.shape, " (토큰 -> 벡터)")
print("W_U shape:", W_U.shape, " (벡터 -> 토큰별 점수)")

## 예제 문장: "나는 오늘 학교에 ___"

이번에는 3개의 토큰으로 이루어진 짧은 문장을 모델에 넣고, 그다음에 올 토큰(4번째 단어)이 무엇일지 예측하게 만들어 봅니다.

```
나는(0)  오늘(1)  학교에(2)  ?
```

한국어 화자라면 자연스럽게 "간다"(토큰 3)를 떠올릴 만한 문맥입니다. 다만 앞서 적었듯 이 모델은 전혀 학습되지 않은 무작위 가중치이기 때문에, "간다"를 예측해 주리라는 보장은 전혀 없습니다. 오히려 이 노트북의 재미는 "모델이 무엇을 예측하든, 왜 그렇게 예측했는지 끝까지 추적할 수 있다"는 데 있습니다. 모델이 똑똑하든 무작위이든, 분해 도구 자체는 똑같이 작동합니다.

모델은 마지막 토큰 위치("학교에"가 들어간 자리)의 잔차 스트림 벡터를 가지고 다음 토큰 점수를 계산합니다. 즉 시퀀스의 **마지막 위치**가 바로 우리가 분석할 예측 지점입니다.

In [ ]:
# -----------------------------------------------------------------
# 입력 시퀀스를 임베딩 벡터로 변환
# -----------------------------------------------------------------
seq = [0, 1, 2]  # "나는", "오늘", "학교에"
seq_len = len(seq)

# W_E[seq] : 정수 배열로 행을 골라내는 numpy의 fancy indexing입니다.
# seq=[0,1,2] 이므로 W_E의 0,1,2번 행(=세 토큰의 임베딩)을 순서대로 쌓아
# (seq_len, d_model) = (3, 12) 모양의 행렬 X를 만듭니다.
X = W_E[seq]
print("입력 문장:", " ".join(id_to_word[t] for t in seq))
print("X shape:", X.shape, " (각 행 = 토큰 하나의 임베딩 벡터)")

# -----------------------------------------------------------------
# 코잘 마스크 (causal mask): 미래 토큰을 보지 못하게 가리는 장치
# -----------------------------------------------------------------
# 언어모델은 '다음 단어 맞히기'를 학습/추론하기 때문에, i번째 위치는
# 자기 자신과 그 이전(0..i) 토큰만 볼 수 있어야 합니다. i보다 뒤에 있는
# (아직 나오지 않은) 토큰을 보고 다음 단어를 맞히는 건 답을 미리 보고
# 푸는 것과 같으므로 반칙입니다.
#
# np.triu(arr, k=1)은 '주대각선보다 한 칸 위쪽(미래 위치)'만 남기고
# 나머지는 0으로 만드는 함수입니다. 그 자리를 -1e9라는 아주 큰 음수로
# 채워두면, 나중에 softmax를 통과할 때 그 위치의 확률이 사실상 0이 되어
# 완전히 무시됩니다 (자세한 이유는 다음 셀의 attn_head 함수에서 확인합니다).
mask = np.triu(np.full((seq_len, seq_len), -1e9), k=1)
print("\nmask (행=쿼리 위치, 열=키 위치):")
print(mask)
print("-> 0이면 '볼 수 있음', -1e9면 '가려짐(미래)'을 의미합니다.")

## 잠깐 복습: Query / Key / Value가 하는 일

본격적으로 attention head를 만들기 전에, Q/K/V의 역할을 도서관 비유로 짧게 복습합니다.

- **Query (질의)**: "나는 지금 무엇을 찾고 있는가?" — 지금 위치의 토큰이 다른 토큰들에게 던지는 질문입니다.
- **Key (열쇠)**: "나는 이런 정보를 갖고 있다" — 각 토큰이 자신을 소개하는 이름표 같은 것입니다. Query와 Key를 내적(dot product)해서 둘이 얼마나 잘 맞는지 점수를 매깁니다.
- **Value (값)**: "내가 실제로 건네줄 내용물" — Query-Key 점수로 만든 가중치(얼마나 주목할지)를 가지고, 각 토큰의 Value를 가중합해서 최종적으로 가져올 정보를 만듭니다.

코드 상에서는 입력 벡터 `X`에 각각 다른 가중치 행렬(`W_Q`, `W_K`, `W_V`)을 곱해서 Q, K, V를 만듭니다. 즉 같은 입력이라도 "질문용으로 가공한 버전(Q)", "이름표용으로 가공한 버전(K)", "실제 내용물 버전(V)"이 서로 다르게 만들어지는 것입니다. 그리고 마지막에 `W_O`를 곱해서, head 내부의 작은 차원(`d_head`)에서 다시 잔차 스트림 차원(`d_model`)으로 되돌립니다.

In [ ]:
# -----------------------------------------------------------------
# attention 가중치 행렬 만들기
# -----------------------------------------------------------------
# 2개 층(layer) x 2개 head 만큼 각각 W_Q, W_K, W_V, W_O를 준비합니다.
# W_Q[l][h] 는 "l번째 층, h번째 head"의 Query 가중치 행렬을 의미합니다.
#
#   W_Q, W_K, W_V : (d_model, d_head)  -- 잔차 스트림(d_model)을 head의
#                                          작은 내부 공간(d_head)으로 투영
#   W_O           : (d_head, d_model)  -- head의 결과를 다시 잔차 스트림
#                                          차원(d_model)으로 복귀
W_Q = [[np.random.randn(d_model, d_head) * 0.2 for _ in range(n_heads)] for _ in range(n_layers)]
W_K = [[np.random.randn(d_model, d_head) * 0.2 for _ in range(n_heads)] for _ in range(n_layers)]
W_V = [[np.random.randn(d_model, d_head) * 0.2 for _ in range(n_heads)] for _ in range(n_layers)]
W_O = [[np.random.randn(d_head, d_model) * 0.2 for _ in range(n_heads)] for _ in range(n_layers)]

print("W_Q[0][0] shape:", W_Q[0][0].shape, " (d_model -> d_head)")
print("W_O[0][0] shape:", W_O[0][0].shape, " (d_head -> d_model)")
print(f"\n총 {n_layers}개 층 x {n_heads}개 head = {n_layers * n_heads}개의 attention head")

In [ ]:
def attn_head(X, wq, wk, wv, wo, mask):
    '''
    attention head 하나를 통째로 계산하는 함수입니다.

    입력
        X          : (seq_len, d_model) 잔차 스트림 (이 head가 읽어들이는 값)
        wq, wk, wv : 이 head의 W_Q, W_K, W_V  (d_model, d_head)
        wo         : 이 head의 W_O  (d_head, d_model)
        mask       : (seq_len, seq_len) 코잘 마스크

    반환
        head_out : (seq_len, d_model) 이 head가 잔차 스트림에 '더할' 값
        A        : (seq_len, seq_len) attention pattern (누가 누구를 얼마나 보는지)
    '''
    # 1) Q, K, V 만들기: X를 각각 다른 행렬에 투영합니다.
    #    q, k, v 모두 shape: (seq_len, d_head)
    #    (변수명을 소문자 q,k,v로 쓴 이유: 이 노트북 앞부분에서 대문자 V를
    #     'vocab_size'라는 의미로 이미 썼기 때문에, 대문자 V를 또 쓰면
    #     Value 벡터인지 어휘 크기인지 헷갈릴 수 있습니다. 이건 함수 안에서만
    #     쓰는 지역 변수라 바깥의 vocab_size와는 전혀 무관하게 동작하지만,
    #     읽는 사람이 헷갈리지 않도록 이름을 다르게 지었습니다.)
    q, k, v = X @ wq, X @ wk, X @ wv

    # 2) attention score 만들기: Query와 Key를 내적해서 '얼마나 잘 맞는지' 점수를 매깁니다.
    #    q @ k.T  -> shape: (seq_len, seq_len), [i, j] = i번째 쿼리와 j번째 키의 내적
    #    sqrt(d_head)로 나누는 이유: d_head가 커질수록 내적 값의 크기도 같이
    #    커지는 경향이 있는데, 그대로 두면 softmax가 한쪽으로 너무 쏠려버립니다
    #    (값 하나만 압도적으로 커서 사실상 토큰 하나만 보게 됨). 차원 수의
    #    제곱근으로 나눠 스케일을 맞춰주는 것이 표준적인 처리입니다.
    #    + mask : 미래 위치에는 -1e9를 더해서 점수를 사실상 -무한대로 만듭니다.
    scores = q @ k.T / np.sqrt(d_head) + mask

    # 3) softmax로 점수를 '합이 1인 가중치(확률)'로 바꿉니다.
    #    -1e9가 더해졌던 자리는 exp(아주 작은 음수) ≈ 0 이 되어 사실상 무시됩니다.
    A = softmax(scores)  # (seq_len, seq_len), 행마다 합이 1

    # 4) A로 V를 가중합한 뒤, W_O로 다시 d_model 차원으로 되돌립니다.
    #    A @ v        : (seq_len, seq_len) @ (seq_len, d_head) -> (seq_len, d_head)
    #    (A @ v) @ wo : (seq_len, d_head) @ (d_head, d_model) -> (seq_len, d_model)
    head_out = (A @ v) @ wo
    return head_out, A

## 전방 패스(forward pass) 진행

이제 준비한 재료(임베딩 `X`, attention 가중치들, 마스크)로 2개 층을 차례로 통과시킵니다. 각 층에서는 다음과 같은 일이 일어납니다.

1. 그 층에 있는 모든 head를 각각 실행합니다.
2. 모든 head의 출력을 더합니다.
3. 그 합을 기존 잔차 스트림에 더합니다 (대체하는 게 아니라 누적됩니다).

즉 `X1 = X + (L0H0 출력) + (L0H1 출력)` 이런 식으로, 층을 통과할수록 잔차 스트림에는 정보가 점점 쌓여 갑니다. 이 누적 구조가 바로 앞에서 말한 "선형 분해가 가능한 이유"입니다.

In [ ]:
# -----------------------------------------------------------------
# Layer 0 (첫 번째 attention 층)
# -----------------------------------------------------------------
h0_outs = []  # 이 층에 있는 각 head의 출력을 따로 저장해 둡니다 (나중에 DLA에서 재사용)
for h in range(n_heads):
    out, A = attn_head(X, W_Q[0][h], W_K[0][h], W_V[0][h], W_O[0][h], mask)
    h0_outs.append(out)
    print(f"[L0H{h}] 마지막 토큰이 보는 attention pattern: {np.round(A[-1], 3)}")

# 두 head의 출력을 더한 뒤, 기존 잔차 스트림 X 위에 더합니다 (대체 X)
X1 = X + sum(h0_outs)
print("\nX1 shape:", X1.shape, " (= X + L0H0 출력 + L0H1 출력)")

In [ ]:
# -----------------------------------------------------------------
# Layer 1 (두 번째 attention 층)
# -----------------------------------------------------------------
# 중요: 이번 입력은 X가 아니라 X1입니다. 즉 layer 1의 head들은
# layer 0이 이미 잔차 스트림에 더해놓은 정보까지 함께 보고 Q/K/V를 만듭니다.
# 이것이 뒤에서 다룰 '합성(composition)'이 가능한 이유입니다.
h1_outs = []
for h in range(n_heads):
    out, A = attn_head(X1, W_Q[1][h], W_K[1][h], W_V[1][h], W_O[1][h], mask)
    h1_outs.append(out)
    print(f"[L1H{h}] 마지막 토큰이 보는 attention pattern: {np.round(A[-1], 3)}")

X2 = X1 + sum(h1_outs)
print("\nX2 shape:", X2.shape, " (= X1 + L1H0 출력 + L1H1 출력)")

# -----------------------------------------------------------------
# 다음 토큰 예측: 마지막 위치의 벡터만 꺼내서 W_U를 곱합니다
# -----------------------------------------------------------------
# X2[-1] : 시퀀스의 마지막 위치("학교에"가 있던 자리)가 최종적으로 갖게 된
#          (d_model,) 크기의 벡터. 다음 토큰 예측은 항상 이 마지막 위치를 봅니다.
logits = X2[-1] @ W_U          # (d_model,) @ (d_model, vocab_size) -> (vocab_size,)
probs = softmax(logits)        # 8개 토큰 각각의 확률
base_pred = int(np.argmax(logits))
base_conf = probs[base_pred]

print("\n[다음 토큰 확률]")
for i in range(vocab_size):
    marker = "  <-- 예측" if i == base_pred else ""
    print(f"   {i} {id_to_word[i]:>4s} : {probs[i]:.4f}{marker}")

print(f"\n모델의 예측: 토큰 {base_pred} ({id_to_word[base_pred]}), 확률 {base_conf:.4f}")
print(f"참고로 '간다'(토큰 3)의 확률: {probs[3]:.4f}")

## Part 1. 직접 로짓 기여 (Direct Logit Attribution, DLA)

방금 모델은 토큰 하나를 예측했습니다. 이제 묻고 싶은 질문은 이것입니다.

> 이 예측 점수(logit)는 직접경로, L0H0, L0H1, L1H0, L1H1 중 어디에서 얼마만큼 나왔을까?

앞서 "핵심 아이디어" 섹션에서 확인했듯, `X2[-1]`은 사실 다섯 개 항의 합입니다.

```
X2[-1] = X[-1] + L0H0_out[-1] + L0H1_out[-1] + L1H0_out[-1] + L1H1_out[-1]
```

그리고 `logits = X2[-1] @ W_U`는 선형 연산이므로, 다섯 개 항을 각각 따로 `W_U`에 곱한 뒤 더해도 결과는 완전히 같습니다.

```
logits = (X[-1]@W_U) + (L0H0_out[-1]@W_U) + (L0H1_out[-1]@W_U) + (L1H0_out[-1]@W_U) + (L1H1_out[-1]@W_U)
```

이렇게 쪼갠 다섯 개의 결과(각각 길이 8짜리 벡터, 8개 토큰 각각에 대한 기여 점수)를 하나씩 뜯어보는 것이 바로 DLA입니다. 두 가지 방식으로 살펴봅니다.

- **norm(크기)으로 보기**: 이 항이 8개 토큰의 점수 전체를 얼마나 강하게 흔드는지, 방향과 상관없이 전체적인 영향력의 크기를 봅니다.
- **부호 있는 값으로 보기**: 모델이 실제로 예측한 토큰 하나의 점수에, 이 항이 플러스(찬성)로 기여했는지 마이너스(반대)로 기여했는지를 봅니다.

In [ ]:
# -----------------------------------------------------------------
# 다섯 개 항을 각각 W_U에 통과시켜서 (vocab_size,) 크기의 '기여 벡터'를 만듭니다
# -----------------------------------------------------------------
direct = X[-1] @ W_U  # 모델이 attention을 전혀 쓰지 않았다면 나왔을 점수 (= '직접경로')

terms = [("직접경로", direct)]
for h in range(n_heads):
    terms.append((f"L0H{h}", h0_outs[h][-1] @ W_U))
for h in range(n_heads):
    terms.append((f"L1H{h}", h1_outs[h][-1] @ W_U))

print("[DLA] 각 항이 8개 토큰 점수 전체에 미치는 영향력의 크기 (norm)")
for name, term in terms:
    print(f"   {name:8s} : ||{np.linalg.norm(term):.4f}||")

# 검증: 다섯 항을 전부 더하면 정말로 원래 logits와 똑같은지 확인
reconstructed = sum(term for _, term in terms)
print("\n[검증] 5개 항의 합 == 원래 logits ?", np.allclose(reconstructed, logits))

직접경로의 norm이 다른 head들보다 훨씬 크게 나왔을 것입니다. 이것만 보고 "직접경로가 가장 중요하다"고 바로 결론 내리기 전에, 한 가지 짚고 넘어갈 점이 있습니다.

**norm의 크기는 가중치 행렬을 몇 번 거치는지에도 영향을 받습니다.** 직접경로는 `W_E -> W_U` 단 두 번의 행렬곱만 거치지만, head 하나의 경로는 `W_V -> W_O -> W_U` 세 번의 행렬곱을 거칩니다. 이 노트북의 가중치들은 0.2~0.3 정도의 작은 표준편차로 초기화되어 있으므로, 행렬을 한 번 더 거칠 때마다 값이 평균적으로 더 작아지는 경향이 생깁니다. 즉 지금 관찰되는 "직접경로가 크다"는 결과에는 모델이 똑똑해서가 아니라 단순히 가중치 초기화 스케일과 경로 길이의 영향도 섞여 있다는 뜻입니다. (실제로 학습이 끝난 모델에서는 이런 스케일이 학습 과정에서 알아서 조정됩니다.)

그래서 norm만으로는 부족하고, 더 직접적인 질문을 던져볼 필요가 있습니다.

> 모델이 실제로 예측한 토큰 하나의 점수에, 각 항이 정확히 얼마를 더하거나 뺐는가?

이번엔 부호(+/-)가 있는 값으로, 예측된 토큰 하나에 대한 기여만 따로 뽑아봅니다.

In [ ]:
# -----------------------------------------------------------------
# 예측된 토큰(base_pred) 하나의 점수에 대한, 항별 '부호 있는' 기여
# -----------------------------------------------------------------
print(f"[DLA] 예측 토큰 {base_pred}번 '{id_to_word[base_pred]}'의 logit에 대한 항별 기여")
running_total = 0.0
for name, term in terms:
    contrib = term[base_pred]            # 8개 점수 중 '예측된 토큰' 자리만 꺼냄
    running_total += contrib
    bar_len = int(abs(contrib) * 200)    # 막대 길이는 비교를 쉽게 하기 위한 시각화용 스케일입니다
    sign = "+" if contrib >= 0 else "-"
    print(f"   {name:8s} {contrib:+.4f}  {sign}{'█' * bar_len}")

print(f"   {'합계':8s} {running_total:+.4f}")
print(f"   (실제 logits[{base_pred}] = {logits[base_pred]:+.4f})")
print("\n[검증] 항별 기여의 합 == logits[예측 토큰] ?", np.isclose(running_total, logits[base_pred]))

## Part 2. 합성 점수 (Composition Score)

DLA는 "각 head가 최종 점수에 얼마나 기여했는가"를 봤습니다. 그런데 head들은 서로 완전히 독립적으로 일하지 않습니다. layer 1의 head는 `X1`(= `X` + layer 0의 출력)을 입력으로 받기 때문에, layer 0이 잔차 스트림에 써놓은 정보를 layer 1이 실제로 읽어서 활용할 가능성이 있습니다. 이렇게 서로 다른 층의 head가 "이어달리기"를 하는 현상을 **합성(composition)** 이라고 부릅니다.

이걸 측정하려면 attention head를 두 개의 작은 "회로(circuit)"로 나눠서 생각하면 편합니다.

- **OV 회로** (`W_V @ W_O`, shape: `(d_model, d_model)`): 이 head가 어떤 토큰에 주목했을 때, 그 정보를 잔차 스트림의 어떤 방향으로 옮겨 적는지를 나타냅니다. ("OV"는 Value를 만들고 Output으로 내보내는 두 단계를 합쳤다는 뜻입니다.)
- **QK 회로**: Query와 Key를 만드는 행렬로, 어떤 입력이 어떤 입력에게 주목할지(attention pattern)를 결정합니다. 이 중에서도 `W_Q`는 "내가 무엇을 찾는지", `W_K`는 "내가 무엇을 가졌다고 내세우는지"를 각각 따로 담당합니다.

이렇게 나누면 합성에는 세 가지 종류가 있을 수 있습니다.

- **Q-합성**: layer 0의 OV 출력이 layer 1 head의 Query 계산(`W_Q`)에 영향을 주는 정도
- **K-합성**: layer 0의 OV 출력이 layer 1 head의 Key 계산(`W_K`)에 영향을 주는 정도
- **V-합성**: layer 0의 OV 출력이 layer 1 head의 Value 계산(`W_V`)에 영향을 주는 정도

세 가지 모두 계산 방식은 같습니다. 이전 head의 OV 회로(`W_V0 @ W_O0`)와 다음 head의 해당 행렬(`W_Q1` 또는 `W_K1` 또는 `W_V1`)을 곱한 뒤, 그 결과가 "이전 head 출력의 크기"와 "다음 head 입력 행렬의 크기"를 곱한 것에 비해 얼마나 큰지를 봅니다.

```
composition_score(A, B) = norm(A @ B) / (norm(A) * norm(B))
```

직관적으로: A와 B가 서로 관련 없는 무작위 방향을 보고 있다면 이 비율은 어떤 기준값(baseline) 근처에 머물 것이고, 만약 학습을 통해 두 head가 실제로 같은 정보(같은 방향)를 주고받도록 정렬되어 있다면 이 비율은 그 기준값보다 뚜렷하게 커질 것입니다.

문제는 "기준값이 정확히 얼마인가"입니다. 짐작으로 숫자를 가져다 쓰는 대신, 이어지는 셀에서 직접 시뮬레이션으로 구해보겠습니다.

In [ ]:
def composition_score(W_A, W_B):
    '''
    두 행렬이 얼마나 같은 방향으로 정렬되어 있는지를, 0 근처(거의 무관)부터
    더 큰 값(강하게 정렬)까지의 숫자 하나로 요약합니다.
        W_A: (d_model, d_model) 형태의 '이전 head의 OV 회로'
        W_B: (d_model, d_head)  형태의 '다음 head의 W_Q 또는 W_K 또는 W_V'
    '''
    composed = W_A @ W_B
    return np.linalg.norm(composed) / (np.linalg.norm(W_A) * np.linalg.norm(W_B) + 1e-8)
    # 1e-8을 더하는 이유: 혹시라도 분모가 0이 되어 계산이 깨지는 것을 막기 위한 안전장치입니다.


print("[합성 점수] layer 0의 head 출력 -> layer 1의 head 입력")
comp_scores = {}  # 나중에 기준선과 비교하기 위해 결과를 저장해 둡니다
for h1 in range(n_heads):
    for h0 in range(n_heads):
        OV0 = W_V[0][h0] @ W_O[0][h0]                  # L0Hh0의 OV 회로: (d_model, d_model)
        q_score = composition_score(OV0, W_Q[1][h1])   # Q-합성
        k_score = composition_score(OV0, W_K[1][h1])   # K-합성
        v_score = composition_score(OV0, W_V[1][h1])   # V-합성
        comp_scores[(h1, h0)] = (q_score, k_score, v_score)
        print(f"   L1H{h1} ← L0H{h0}  :  Q-합성={q_score:.4f}   K-합성={k_score:.4f}   V-합성={v_score:.4f}")

### 이 점수가 큰 건지 작은 건지, 어떻게 판단할까?

위 숫자들만 보면 0.2~0.3이 큰 건지 작은 건지 감이 잘 오지 않습니다. 기준이 없는 숫자는 해석하기 어렵습니다.

가장 확실한 기준을 만드는 방법은, "두 head가 서로 전혀 관련 없이 완전히 무작위로 초기화됐다면 이 점수가 평균적으로 얼마나 나올까?"를 직접 여러 번 시뮬레이션해서 알아내는 것입니다. 지금 우리가 쓰고 있는 `W_V[0][h0]`, `W_O[0][h0]`, `W_K[1][h1]` 같은 행렬들도 사실 전부 학습되지 않은 무작위 값이므로, 이 기준선과 비교하면 지금 관찰한 점수가 정말로 '무작위 수준'인지 바로 확인할 수 있습니다.

In [ ]:
# -----------------------------------------------------------------
# 무작위 기준선을 시뮬레이션으로 추정하기
# -----------------------------------------------------------------
# np.random.seed(42)로 고정해둔 전역 난수 상태를 건드리지 않기 위해,
# 별도의 독립적인 난수 생성기(rng)를 새로 만들어서 사용합니다.
rng = np.random.default_rng(123)

n_trials = 3000
sim_scores = []
for _ in range(n_trials):
    # 우리 모델과 완전히 같은 shape, 같은 초기화 스케일(0.2)을 가졌지만
    # 서로 아무 관계도 없는 '진짜 무작위' 행렬들을 새로 뽑습니다.
    wv = rng.standard_normal((d_model, d_head)) * 0.2
    wo = rng.standard_normal((d_head, d_model)) * 0.2
    wk = rng.standard_normal((d_model, d_head)) * 0.2
    sim_scores.append(composition_score(wv @ wo, wk))
sim_scores = np.array(sim_scores)

baseline_mean = sim_scores.mean()
baseline_std = sim_scores.std()
print(f"[무작위 기준선] {n_trials}번 시뮬레이션 결과")
print(f"   평균: {baseline_mean:.4f}")
print(f"   표준편차: {baseline_std:.4f}")
print(f"   1 표준편차 범위: [{baseline_mean - baseline_std:.4f}, {baseline_mean + baseline_std:.4f}]")

print("\n[비교] 우리 모델에서 실제로 관찰한 점수들이 이 범위 안에 들어오는지 확인해 보세요.")
for (h1, h0), (q_s, k_s, v_s) in comp_scores.items():
    print(f"   L1H{h1} ← L0H{h0}  :  Q={q_s:.4f}  K={k_s:.4f}  V={v_s:.4f}")

### 결과 해석

위에서 출력된 평균/표준편차 범위와 실제 관찰값을 비교해 보면, 이 노트북의 합성 점수들은 대체로 기준선 범위 안에(또는 아주 가까이) 들어오는 것을 확인할 수 있습니다. 이건 우연이 아니라 당연한 결과입니다. 애초에 이 모델의 모든 가중치가 `np.random.seed(42)`로 뽑은 무작위 값이고, **학습을 한 번도 하지 않았기** 때문입니다. 즉 layer 0과 layer 1의 head들 사이에 의도적으로 정보를 주고받는 관계가 생길 이유가 전혀 없습니다.

그렇다면 실제로 학습된 모델에서는 무엇이 달라질까요? 대표적인 예시가 **유도 헤드(induction head)** 입니다. 트랜스포머를 학습시키면, 종종 이런 두 head 조합이 만들어집니다.

- 앞쪽 층의 한 head가 "바로 직전 토큰이 무엇이었는지"를 OV 회로를 통해 잔차 스트림에 적어 둡니다 (previous-token head).
- 뒤쪽 층의 한 head가 K-합성을 통해 이 정보를 읽어서, 지금 토큰과 똑같은 토큰이 과거에 나왔던 위치를 찾고, 그다음에 이어졌던 토큰을 그대로 복사해서 예측합니다 (induction head).

이런 head 쌍은 K-합성 점수가 무작위 기준선보다 훨씬 높게 나옵니다. 두 head가 우연이 아니라 실제로 서로의 출력을 활용하도록 학습되었기 때문입니다. (이 주제는 Anthropic의 트랜스포머 회로(transformer circuits) 해석가능성 연구에서 자세히 다뤄졌습니다. 더 깊이 알고 싶다면 "induction head"를 키워드로 찾아보는 것을 추천합니다.)

즉 합성 점수 자체는 모델이 학습됐는지 아닌지와 무관하게 항상 계산할 수 있는 도구이고, 그 값이 무작위 기준선보다 뚜렷하게 높을 때 비로소 "이 두 head는 실제로 협력하도록 학습되었다"고 해석할 수 있습니다.

## Part 3. 어블레이션 (Ablation)

DLA는 "이 항이 단독으로 얼마나 큰 기여를 했는가"를 봤습니다. 그런데 이건 어디까지나 상관관계에 가깝습니다. 더 직접적으로 "이 항이 정말로 없으면 안 되는가?"를 알고 싶다면, 실제로 그 항을 제거해 보고 결과가 얼마나 바뀌는지 관찰하는 것이 가장 확실합니다. 이런 식으로 구성 요소를 일부러 망가뜨려보고 그 효과를 관찰하는 실험을 어블레이션(ablation)이라고 부릅니다. 의학에서 "조직을 절제해서 그 기능을 알아낸다"는 표현에서 따온 용어입니다.

이 노트북에서는 가장 단순한 형태의 어블레이션을 사용합니다. 이미 다 계산해 둔 logits에서 항 하나를 그냥 빼버리는 방식입니다.

```
ablated_logits = logits - (제거하고 싶은 항)
```

그런 다음 원래 예측했던 토큰의 확률이 얼마나 떨어지는지(`Δ`)를 봅니다. `Δ`가 크면 그 항이 이 예측에 정말 중요했다는 뜻이고, `Δ`가 거의 0이면 그 항을 빼도 예측이 별로 흔들리지 않았다는 뜻입니다.

**주의할 점**: 이 방식은 정확히 말하면 "직접 효과만" 제거하는 어블레이션입니다. 예를 들어 L0H0을 이런 식으로 "제거"해도, 실제로는 layer 1의 head들이 attention pattern을 계산할 때 사용한 `X1`(= L0H0의 출력이 이미 포함된 값)은 전혀 건드리지 않습니다. 즉 L0H0이 layer 1에 미쳤을 간접적인 영향(= 합성을 통한 영향)은 이 계산에 전혀 반영되지 않습니다. 이 차이를 직접 눈으로 확인하기 위해, 이번 섹션 뒤쪽에서 "진짜로 L0 head를 0으로 만들고 layer 1을 처음부터 다시 계산하는" 버전과 비교해 봅니다.

In [ ]:
# -----------------------------------------------------------------
# 어블레이션: 항을 하나씩 빼면서 예측 확률이 얼마나 떨어지는지 관찰
# -----------------------------------------------------------------
print(f"[Ablation] 기준 예측: 토큰 {base_pred} ({id_to_word[base_pred]}), 확률 {base_conf:.4f}\n")

for name, term in terms:
    ablated_logits = logits - term                      # 이 항만 쏙 빼기
    ablated_conf = softmax(ablated_logits)[base_pred]    # 같은 토큰(base_pred)의 확률을 다시 계산
    delta = base_conf - ablated_conf                     # 확률이 얼마나 떨어졌는가
    bar = "█" * max(0, int(abs(delta) * 300))
    print(f"   {name:8s} 제거 -> 확률 {ablated_conf:.4f}   Δ={delta:+.4f}  {bar}")

### "진짜로" 제거하면 어떨까: 직접 효과 vs 전체 효과

L0 layer의 head는 L1 layer 입장에서 입력의 일부이기도 합니다. 그래서 L0H0을 정말로 제거한다면, 다음 순서로 처음부터 다시 계산해야 정확합니다.

1. L0H0의 출력을 0으로 만듭니다.
2. `X1`을 다시 계산합니다 (L0H0의 출력 없이).
3. `X1`이 바뀌었으니, layer 1의 두 head도 새로운 `X1`을 가지고 attention을 처음부터 다시 계산합니다.
4. 그렇게 나온 새로운 `X2`로 최종 logits을 다시 계산합니다.

이렇게 처음부터 다시 계산한 효과를 **전체 효과(total effect)**, 앞서 했던 "그냥 항만 빼는" 방식을 **직접 효과(direct effect)** 라고 구분해서 부릅니다. 둘의 차이가 바로 L0H0이 layer 1을 거쳐서 미친 **간접 효과(indirect effect)** 입니다. L1 head들은 그 뒤에 더 이상 층이 없으므로(이 노트북은 2층짜리 모델입니다), L1 head에 대해서는 직접 효과와 전체 효과가 항상 같습니다. 그래서 비교는 L0 head에 대해서만 의미가 있습니다.

In [ ]:
# -----------------------------------------------------------------
# L0 head를 '직접 효과만' 제거했을 때 vs '전체를 다시 계산'했을 때 비교
# -----------------------------------------------------------------
print("[Ablation 비교] L0 head: direct effect vs total effect\n")

for h0 in range(n_heads):
    # (1) 직접 효과만: 앞에서 이미 계산한 것과 같은 방식
    term = h0_outs[h0][-1] @ W_U
    direct_conf = softmax(logits - term)[base_pred]
    direct_delta = base_conf - direct_conf

    # (2) 전체 효과: L0Hh0의 출력을 진짜로 0으로 만들고 layer 1부터 다시 계산
    h0_outs_ablated = [
        np.zeros_like(h0_outs[i]) if i == h0 else h0_outs[i]
        for i in range(n_heads)
    ]
    X1_ablated = X + sum(h0_outs_ablated)   # L0Hh0이 아예 없었다면 만들어졌을 X1

    h1_outs_ablated = []
    for h in range(n_heads):
        out, _ = attn_head(X1_ablated, W_Q[1][h], W_K[1][h], W_V[1][h], W_O[1][h], mask)
        h1_outs_ablated.append(out)
    X2_ablated = X1_ablated + sum(h1_outs_ablated)

    logits_full = X2_ablated[-1] @ W_U
    full_conf = softmax(logits_full)[base_pred]
    full_delta = base_conf - full_conf

    indirect = full_delta - direct_delta
    print(f"   L0H{h0}:  direct-only Δ={direct_delta:+.4f}   total Δ={full_delta:+.4f}   간접효과={indirect:+.4f}")

print("\n간접효과가 0에서 멀리 떨어져 있을수록, 그 head가 layer 1의 attention pattern을 바꾸는 방식으로")
print("'직접 보이지 않는' 영향을 미치고 있다는 뜻입니다.")

## 정리

이 노트북에서는 같은 질문 — "이 예측은 왜 나왔는가?" — 을 세 가지 다른 각도에서 다뤄봤습니다.

- **DLA**: 잔차 스트림이 여러 항의 합이라는 사실을 이용해서, 최종 점수를 각 구성 요소의 기여로 정확하게(근사가 아니라 정확하게) 쪼갰습니다.
- **합성 점수**: 한 head의 출력을 다른 head가 실제로 읽어서 쓰고 있는지를, 무작위 기준선과 비교해서 측정했습니다.
- **어블레이션**: 구성 요소를 실제로 제거해 보면서, 그 구성 요소가 인과적으로 얼마나 필요했는지 확인했습니다. 그리고 직접 효과와 전체 효과가 다를 수 있다는 것도 직접 비교해 봤습니다.

### 직접 해보기

아래와 같이 값을 바꿔가며 다시 실행해 보면 이해에 도움이 됩니다.

- `np.random.seed(42)`를 다른 숫자로 바꿔보고, DLA/합성/어블레이션 결과가 어떻게 달라지는지 비교해 보세요.
- `seq`를 다른 토큰 조합으로 바꿔서, 다른 예측에 대해 같은 분석을 반복해 보세요.
- `n_heads`나 `d_head`를 바꿔보고, 합성 점수의 무작위 기준선(평균/표준편차)이 각각 어떻게 달라지는지 비교해 보세요. (힌트: 기준선의 '평균'과 '흩어진 정도(표준편차)'는 서로 다른 요인에 더 민감하게 반응합니다.)
- L1 head도 위와 같은 방식으로 "직접 효과 vs 전체 효과"를 비교할 수 있을지 생각해 보세요. (힌트: 이 노트북이 몇 층짜리 모델인지 떠올려 보세요.)